# Pitch Vision — Phase 0 & 1 (v2 — drone-matched training data)

**What changed from v1:** the first fine-tune used a Roboflow dataset shot from broadcast-style cameras (players large, medium shot). Our footage is a straight-down drone view, where players are tiny — a few pixels each. That domain mismatch is why the v1 model detected almost nothing correctly.

This version fine-tunes on frames pulled directly from our own SoccerTrack drone clips, using SoccerTrack's own per-frame bounding-box ground truth (converted to YOLO format). Two classes only: `player` and `ball` — team split (and filtering out the referee, if needed) happens later via K-means color clustering in Stage 2, same as the roadmap already planned.

**Before running:** Runtime → Change runtime type → GPU (T4, free tier).

You'll need: `soccertrack_yolo_dataset.zip` from your project's `training/` folder (built locally from the SoccerTrack CSVs — ~525 labeled drone frames, 1920x1080, players + ball).

In [ ]:
!pip install -q ultralytics supervision
from ultralytics import YOLO
import ultralytics
ultralytics.checks()

## Part A — Baseline (optional, same as before)

Skip this if you already have `baseline_output.zip` from the v1 run — it's the same stock-YOLO comparison, doesn't depend on anything below. Only run it if you want to redo it fresh.

In [ ]:
from google.colab import files
print("Upload clip_01_0-12s.mp4 (from your project's input_videos/ folder):")
uploaded = files.upload()
clip_path = list(uploaded.keys())[0]
print("Using:", clip_path)

## Part B — Fine-tune YOLO on drone-matched player/ball data

Trains on frames extracted straight from your SoccerTrack top-view clips, labeled from the dataset's own ground-truth CSVs (`bb_left`/`bb_top`/`bb_width`/`bb_height` per player and ball, every frame). 7 clips spread across the match, every 12th frame, 2 classes: `player`, `ball`.

In [ ]:
from google.colab import files
print("Upload soccertrack_yolo_dataset.zip (from your project's training/ folder):")
uploaded_ds = files.upload()

In [ ]:
import zipfile, os

zip_name = [k for k in uploaded_ds.keys() if k.endswith('.zip')][0]
extract_dir = '/content/soccertrack_yolo_dataset'
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(extract_dir)

train_dir = os.path.join(extract_dir, 'images', 'train')
val_dir = os.path.join(extract_dir, 'images', 'valid')
print("train exists:", os.path.isdir(train_dir), "-", len(os.listdir(train_dir)) if os.path.isdir(train_dir) else 0, "images")
print("valid exists:", os.path.isdir(val_dir), "-", len(os.listdir(val_dir)) if os.path.isdir(val_dir) else 0, "images")

# Rewrite data.yaml with the actual Colab-side paths (this is exactly the bug that bit us
# last time with the Roboflow download — the paths inside a downloaded/extracted data.yaml
# don't match wherever it lands on THIS machine, so we always regenerate it after extracting).
data_yaml = os.path.join(extract_dir, 'data.yaml')
with open(data_yaml, 'w') as f:
    f.write(f"train: {train_dir}\nval: {val_dir}\nnc: 2\nnames: ['player', 'ball']\n")
print("data.yaml at:", data_yaml)
!cat {data_yaml}

In [ ]:
# Same settings that already survived a full 100-epoch run without crashing (yolov8m, batch=8).
# If Colab disconnects mid-run, re-run this cell with resume=True to continue from the last checkpoint.
!yolo task=detect mode=train model=yolov8m.pt data="{data_yaml}" epochs=100 imgsz=640 batch=8 workers=2

In [ ]:
# Grab the trained weights — this is the one file you need to bring back to your local project's models/ folder.
import glob, shutil
best_pt = glob.glob('runs/detect/train*/weights/best.pt')[-1]
shutil.copy(best_pt, 'best.pt')
files.download('best.pt')
print("Downloaded best.pt — save this into your local project's models/ folder (this replaces the v1 best.pt).")

## Validate: does the fine-tuned model actually find players and the ball now?

Re-run on the exact same clip. This time expect real `player` boxes on the actual dots, not scattered noise labeled `ball`.

In [ ]:
from ultralytics import YOLO
import glob, shutil
from google.colab import files

finetuned_model = YOLO('best.pt')

class_counts = {}
total_ft_frames = 0

# stream=True processes and discards each frame's result as it goes, instead of holding
# the whole video's detections in RAM at once — this is what fixed the crash from before.
for r in finetuned_model.predict(source=clip_path, save=True, conf=0.25, stream=True):
    total_ft_frames += 1
    for c in r.boxes.cls:
        name = finetuned_model.names[int(c)]
        class_counts[name] = class_counts.get(name, 0) + 1

print("Classes this model knows:", finetuned_model.names)
print(f"\nTotal frames: {total_ft_frames}")
for name, count in class_counts.items():
    print(f"  {name}: {count} detections total, {count/total_ft_frames:.1f} avg/frame")

out_dir = glob.glob('runs/detect/predict*')[-1]
shutil.make_archive('finetuned_output_v2', 'zip', out_dir)
files.download('finetuned_output_v2.zip')

### Done with Colab for now

You should have downloaded: `best.pt`, `finetuned_output_v2.zip` (and `baseline_output.zip` if you ran Part A).

Put `best.pt` in your local project's `models/` folder, replacing the v1 version — that's what Stage 2 (tracking) will load.

Note: `conf=0.25` here (higher than the `0.1` used in v1) — now that the model is actually trained on the right kind of imagery, a higher confidence threshold makes more sense for judging real quality instead of drowning in low-confidence noise.